# Treino DeepLabV3+ no Kaggle (GPU) — pré-treino, linhas e multiclasse

Objetivo: modelo que identifica **telhado** (incl. quando casas estão grudadas), **divisorias** entre telhados, **águas**, **claraboias** e **lajes**.

**Classes (chips_multiclass):** 0=fundo | 1=águas (planos do telhado) | 2=claraboia | 3=**divisória** (fronteira entre telhados de casas diferentes) | 4=laje. As máscaras devem marcar bem as **divisorias** para separar casas adjacentes.

1. **roof/chips** (binário) → pré-treino → `deeplabv3_roof_pretrain.pt`
2. **roof/chips_segmentos** → modelo de linhas (U-Net) → `unet_lines.pt`
3. **roof/chips_multiclass** (5 classes) → fine-tuning → `deeplabv3_roof_multiclass.pt`

**Input Kaggle:** ex. `/kaggle/input/datasets/paulohenriquemota/roof-dataset` com pasta **roof/** (e dentro `roof/chips_multiclass/images/`, `roof/chips_multiclass/masks/`).

**Saída:** `deeplabv3_roof_multiclass.pt`, opcional `unet_lines.pt` e `deeplabv3_roof_pretrain.pt` em `/kaggle/working/models/`.

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
USE_GPU = (out.returncode == 0)
if USE_GPU:
    print('GPU:', out.stdout.strip())
else:
    print('Nenhuma GPU. Ativa em Settings → Accelerator → GPU.')

In [ ]:
import os
REPO_URL = "https://github.com/phmotad/roofArea.git"
REPO_DIR = "/kaggle/working/roofArea"

if os.path.isfile(os.path.join(REPO_DIR, "pyproject.toml")) and os.path.isdir(os.path.join(REPO_DIR, "src", "roof_api")):
    print("Projeto já presente em", REPO_DIR)
else:
    if os.path.isdir(REPO_DIR):
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Diretório atual:", os.getcwd())

In [ ]:
subprocess.run(["pip", "install", "-q", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run(["pip", "install", "-q", "pillow", "numpy", "opencv-python-headless", "scikit-image"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Dependências instaladas.")

**Caminhos:** Detetar em `/kaggle/input/` pasta **roof/** (chips + chips_multiclass + chips_segmentos) ou datasets separados. Se não existir roof/, construímos em `/kaggle/working/roof`.

In [ ]:
import os
KAGGLE_INPUT = "/kaggle/input"

def _has_subdirs(root, *subdirs):
    return all(os.path.isdir(os.path.join(root, d)) for d in subdirs)

def _has_files_in(dirpath, exts=(".png", ".jpg", ".jpeg")):
    if not os.path.isdir(dirpath):
        return False
    try:
        names = os.listdir(dirpath)
        return any(f.lower().endswith(ext) for f in names[:50] for ext in exts)
    except OSError:
        return False

roots_with_name = []
def add_roots(base_path, depth=0):
    if depth > 2:
        return
    try:
        for name in os.listdir(base_path):
            path = os.path.join(base_path, name)
            if os.path.isdir(path):
                roots_with_name.append((path, name.lower()))
                add_roots(path, depth + 1)
    except OSError:
        pass
add_roots(KAGGLE_INPUT)

CHIPS_MC_PATH = ""
CHIPS_BIN_PATH = ""
SEG_DIR = ""
ROOF_DIR = ""
INRIA_PATH = ""
ROOFSAT_PATH = ""
BUILD_ROOF = False

# Caminho explícito do dataset roof-dataset no Kaggle
ROOF_DATASET_INPUT = "/kaggle/input/datasets/paulohenriquemota/roof-dataset"
if os.path.isdir(ROOF_DATASET_INPUT):
    p = os.path.join(ROOF_DATASET_INPUT, "roof", "chips_multiclass")
    if os.path.isdir(p) and _has_subdirs(p, "images", "masks") and _has_files_in(os.path.join(p, "images")):
        CHIPS_MC_PATH = p
        ROOF_DIR = os.path.dirname(p)
        CHIPS_BIN_PATH = os.path.join(ROOF_DIR, "chips")
        seg = os.path.join(ROOF_DIR, "chips_segmentos")
        SEG_DIR = seg if os.path.isdir(seg) and _has_files_in(os.path.join(seg, "images")) else ""

# Preferir roof/ completo (chips + chips_multiclass) se ainda não temos CHIPS_MC_PATH
for root, _ in roots_with_name:
    if _has_subdirs(root, "chips", "chips_multiclass") and _has_files_in(os.path.join(root, "chips", "images")) and _has_files_in(os.path.join(root, "chips_multiclass", "images")):
        ROOF_DIR = root
        CHIPS_MC_PATH = os.path.join(ROOF_DIR, "chips_multiclass")
        CHIPS_BIN_PATH = os.path.join(ROOF_DIR, "chips")
        seg = os.path.join(ROOF_DIR, "chips_segmentos")
        SEG_DIR = seg if os.path.isdir(seg) and _has_files_in(os.path.join(seg, "images")) else ""
        BUILD_ROOF = False
        break

def _check_chips_mc(root):
    for sub in (os.path.join(root, "roof", "chips_multiclass"), os.path.join(root, "chips_multiclass"), root):
        if os.path.isdir(sub) and _has_subdirs(sub, "images", "masks") and _has_files_in(os.path.join(sub, "images")):
            return sub
    return ""
def _check_inria(root, exclude=""):
    c = os.path.join(root, "dados_inria") if os.path.isdir(os.path.join(root, "dados_inria")) else root
    if _has_subdirs(c, "images", "masks") and _has_files_in(os.path.join(c, "images")) and c != exclude:
        return c
    return ""
def _check_roofsat(root):
    rs = os.path.join(root, "Roofsat") if os.path.isdir(os.path.join(root, "Roofsat")) else root
    if os.path.isdir(os.path.join(rs, "building_masks")) and os.path.isfile(os.path.join(rs, "train.txt")):
        return rs
    return ""

if not CHIPS_MC_PATH:
    for root, name in roots_with_name:
        if ("chip" in name or "multiclass" in name or "roof" in name) and not CHIPS_MC_PATH:
            c = _check_chips_mc(root)
            if c:
                CHIPS_MC_PATH = c
                # Se chips_multiclass está dentro de .../roof/, usar esse roof/ para chips e segmentos
                parent = os.path.dirname(c)
                if os.path.basename(parent) == "roof":
                    ROOF_DIR = parent
                    CHIPS_BIN_PATH = os.path.join(ROOF_DIR, "chips")
                    seg = os.path.join(ROOF_DIR, "chips_segmentos")
                    SEG_DIR = seg if os.path.isdir(seg) and _has_files_in(os.path.join(seg, "images")) else ""
        if "inria" in name and not INRIA_PATH:
            i = _check_inria(root, CHIPS_MC_PATH)
            if i: INRIA_PATH = i
        if "roofsat" in name and not ROOFSAT_PATH:
            r = _check_roofsat(root)
            if r: ROOFSAT_PATH = r
    for root, _ in roots_with_name:
        if not CHIPS_MC_PATH: CHIPS_MC_PATH = _check_chips_mc(root)
        if not INRIA_PATH: INRIA_PATH = _check_inria(root, CHIPS_MC_PATH)
        if not ROOFSAT_PATH: ROOFSAT_PATH = _check_roofsat(root)
        if CHIPS_MC_PATH and not ROOF_DIR:
            parent = os.path.dirname(CHIPS_MC_PATH)
            if os.path.basename(parent) == "roof":
                ROOF_DIR = parent
                CHIPS_BIN_PATH = os.path.join(ROOF_DIR, "chips")
                seg = os.path.join(ROOF_DIR, "chips_segmentos")
                SEG_DIR = seg if os.path.isdir(seg) and _has_files_in(os.path.join(seg, "images")) else ""
    try:
        for top in os.listdir(KAGGLE_INPUT):
            base = os.path.join(KAGGLE_INPUT, top)
            if not os.path.isdir(base): continue
            if not CHIPS_MC_PATH:
                p = os.path.join(base, "roof-chips-multiclass", "chips_multiclass")
                if os.path.isdir(p) and _has_subdirs(p, "images", "masks"): CHIPS_MC_PATH = p
            if not CHIPS_MC_PATH:
                p = os.path.join(base, "roof-dataset", "roof", "chips_multiclass")
                if os.path.isdir(p) and _has_subdirs(p, "images", "masks"):
                    CHIPS_MC_PATH = p
                    ROOF_DIR = os.path.dirname(p)
                    CHIPS_BIN_PATH = os.path.join(ROOF_DIR, "chips")
                    SEG_DIR = os.path.join(ROOF_DIR, "chips_segmentos")
            if not INRIA_PATH:
                ip = os.path.join(base, "roof-inria-patches")
                if _has_subdirs(ip, "images", "masks") and _has_files_in(os.path.join(ip, "images")): INRIA_PATH = ip
            if not ROOFSAT_PATH:
                rp = os.path.join(base, "roof-roofsat", "Roofsat")
                if os.path.isdir(rp) and os.path.isdir(os.path.join(rp, "building_masks")) and os.path.isfile(os.path.join(rp, "train.txt")): ROOFSAT_PATH = rp
    except OSError:
        pass
    if not ROOF_DIR: ROOF_DIR = "/kaggle/working/roof"
    BUILD_ROOF = True
    ROOF_DIR = "/kaggle/working/roof"

if not CHIPS_MC_PATH:
    raise SystemExit("Nenhum dataset chips_multiclass em /kaggle/input/. Anexa um dataset com roof/chips_multiclass ou roof-chips-multiclass.")

if BUILD_ROOF and (INRIA_PATH or ROOFSAT_PATH):
    CHIPS_BIN_PATH = os.path.join(ROOF_DIR, "chips")
    SEG_DIR = os.path.join(ROOF_DIR, "chips_segmentos")

print("Chips multiclass:", CHIPS_MC_PATH)
print("Chips (binário, pré-treino):", CHIPS_BIN_PATH if CHIPS_BIN_PATH and os.path.isdir(os.path.join(CHIPS_BIN_PATH, "images")) else "—")
print("Linhas (chips_segmentos):", SEG_DIR if SEG_DIR and os.path.isdir(SEG_DIR) else "—")
print("ROOF_DIR:", ROOF_DIR, "(construir?)" if BUILD_ROOF else "(já existe)")
if INRIA_PATH: print("Inria:", INRIA_PATH)
if ROOFSAT_PATH: print("RoofSat:", ROOFSAT_PATH)

In [ ]:
# Construir roof/ a partir dos datasets (se BUILD_ROOF)
import subprocess
if BUILD_ROOF:
    inria_for_roof = INRIA_PATH if INRIA_PATH and os.path.isdir(os.path.join(INRIA_PATH, "images")) else ""
    if not inria_for_roof:
        inria_for_roof = "/kaggle/working/dados_inria"
        print("A descarregar Inria para", inria_for_roof, "...")
        subprocess.run(["python", "-m", "scripts.download_inria_dataset", "--output_dir", inria_for_roof], check=True)
    cmd = ["python", "-m", "scripts.prepare_roof_structure", "--output", ROOF_DIR, "--inria", inria_for_roof, "--chips_multiclass", CHIPS_MC_PATH]
    if ROOFSAT_PATH: cmd.extend(["--roofsat", ROOFSAT_PATH])
    subprocess.run(cmd, check=True)
    CHIPS_MC_PATH = os.path.join(ROOF_DIR, "chips_multiclass")
    CHIPS_BIN_PATH = os.path.join(ROOF_DIR, "chips")
    SEG_DIR = os.path.join(ROOF_DIR, "chips_segmentos")
else:
    print("roof/ já presente; a ignorar construção.")

**Pré-treino binário** (roof/chips): treina DeepLabV3+ com 1 classe (telhado vs fundo). Opcional se roof/chips existir.

In [ ]:
# Pré-treino binário DeepLabV3+ (roof/chips)
import sys
import os
import torch
from pathlib import Path
from torch.utils.data import DataLoader, random_split
from torchvision.models.segmentation import deeplabv3_resnet50

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from roof_api.segmentation.dataset import RoofDataset

CHIPS_BIN_IMAGES = Path(CHIPS_BIN_PATH) / "images" if CHIPS_BIN_PATH else None
CHIPS_BIN_MASKS = Path(CHIPS_BIN_PATH) / "masks" if CHIPS_BIN_PATH else None
has_bin = CHIPS_BIN_IMAGES and CHIPS_BIN_IMAGES.is_dir() and CHIPS_BIN_MASKS and CHIPS_BIN_MASKS.is_dir()

if has_bin:
    os.makedirs("/kaggle/working/models", exist_ok=True)
    SIZE = (512, 512)
    device = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")
    ds_bin = RoofDataset(CHIPS_BIN_IMAGES, CHIPS_BIN_MASKS, size=SIZE, augment=True, num_classes=1)
    n_bin = len(ds_bin)
    n_val_bin = max(1, min(50, n_bin // 5))
    n_train_bin = n_bin - n_val_bin
    train_bin, val_bin = random_split(ds_bin, [n_train_bin, n_val_bin], generator=torch.Generator().manual_seed(42))
    load_bin = DataLoader(train_bin, batch_size=4, shuffle=True, num_workers=0)
    load_val_bin = DataLoader(val_bin, batch_size=4, shuffle=False, num_workers=0)
    model_pretrain = deeplabv3_resnet50(num_classes=1, weights=None, weights_backbone=None).to(device)
    opt = torch.optim.AdamW(model_pretrain.parameters(), lr=1e-4)
    crit = torch.nn.BCEWithLogitsLoss()
    for epoch in range(1, 31):
        model_pretrain.train()
        loss_sum = 0.0
        for x, y in load_bin:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model_pretrain(x)["out"]
            loss = crit(out, y.float())
            loss.backward()
            opt.step()
            loss_sum += loss.item() * x.size(0)
        print(f"Pretrain epoch {epoch} loss={loss_sum/n_train_bin:.4f}")
    torch.save({"model": model_pretrain.state_dict(), "num_classes": 1}, "/kaggle/working/models/deeplabv3_roof_pretrain.pt")
    print("Guardado deeplabv3_roof_pretrain.pt")
else:
    print("roof/chips sem images/masks; pré-treino ignorado.")

**Modelo de linhas** (roof/chips_segmentos): gera máscaras a partir dos .npz e treina U-Net → `unet_lines.pt` (linhas das águas e divisorias). Opcional.

In [ ]:
# Modelo de linhas (U-Net em chips_segmentos)
import subprocess
import os
SEG_DIR_CHECK = os.path.join(SEG_DIR, "gt") if SEG_DIR else ""
if SEG_DIR and os.path.isdir(SEG_DIR):
    if os.path.isdir(SEG_DIR_CHECK) and os.path.isdir(os.path.join(SEG_DIR, "images")):
        subprocess.run(["python", "-m", "scripts.rasterize_chips_segmentos_masks", "--segmentos_dir", SEG_DIR], check=True, cwd=REPO_DIR)
    masks_seg = os.path.join(SEG_DIR, "masks")
    if os.path.isdir(masks_seg) and len(os.listdir(masks_seg)) > 0:
        subprocess.run([
            "python", "-u", "-m", "scripts.train_unet",
            "--data_dir", SEG_DIR, "--output", "/kaggle/working/models/unet_lines.pt",
            "--num_classes", "1", "--size", "512", "512", "--epochs", "40", "--device", "cuda" if USE_GPU else "cpu"
        ], check=True, cwd=REPO_DIR)
        print("Guardado unet_lines.pt")
    else:
        print("chips_segmentos sem máscaras; modelo de linhas ignorado.")
else:
    print("chips_segmentos não encontrado; modelo de linhas ignorado.")

**Fine-tuning multiclasse** em roof/chips_multiclass (5 classes). Carrega pré-treino binário se existir.

In [ ]:
import sys
import os
import torch
from pathlib import Path
from torch.utils.data import DataLoader, random_split

# Pacote está em src/roof_api; garantir que o kernel encontra
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from roof_api.segmentation.dataset import RoofDataset

NUM_CLASSES = 5
SIZE = (512, 512)
BATCH_SIZE = 4
VAL_RATIO = 0.2
device = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")
print("Device:", device)

images_dir = Path(CHIPS_MC_PATH) / "images"
masks_dir = Path(CHIPS_MC_PATH) / "masks"
full_ds = RoofDataset(images_dir, masks_dir, size=SIZE, augment=True, num_classes=NUM_CLASSES)
n = len(full_ds)
n_val = max(1, int(n * VAL_RATIO))
n_train = n - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))
print("Train:", n_train, "Val:", n_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50

model = deeplabv3_resnet50(num_classes=NUM_CLASSES, weights=None, weights_backbone=None)
PRETRAIN_PATH = "/kaggle/working/models/deeplabv3_roof_pretrain.pt"
if os.path.isfile(PRETRAIN_PATH):
    ckpt = torch.load(PRETRAIN_PATH, map_location=device, weights_only=True)
    sd = ckpt.get("model", ckpt)
    model_sd = model.state_dict()
    filtered = {k: v for k, v in sd.items() if k in model_sd and v.shape == model_sd[k].shape}
    model.load_state_dict(filtered, strict=False)
    print("Carregado pré-treino:", PRETRAIN_PATH, "(%d/%d parâmetros)" % (len(filtered), len(sd)))
model = model.to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print("Modelo DeepLabV3 ResNet50, num_classes=", NUM_CLASSES)

In [ ]:
EPOCHS = 30
OUTPUT_PATH = "/kaggle/working/models/deeplabv3_roof_multiclass.pt"
os.makedirs("/kaggle/working/models", exist_ok=True)

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        logits = out["out"]
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= n_train

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            logits = out["out"]
            val_loss += criterion(logits, y).item() * x.size(0)
    val_loss /= n_val

    print(f"Epoch {epoch} | train loss={train_loss:.4f} | val loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"model": model.state_dict(), "num_classes": NUM_CLASSES}, OUTPUT_PATH)
        print(f"  -> Guardado melhor modelo em {OUTPUT_PATH}")

print("Treino concluído. Melhor modelo:", OUTPUT_PATH)

In [ ]:
import os
for f in sorted(os.listdir("/kaggle/working/models")):
    path = os.path.join("/kaggle/working/models", f)
    print(f, "—", os.path.getsize(path) // 1024, "KB")
print("\nDescarrega os .pt para models/: deeplabv3_roof_multiclass.pt (API), opcional unet_lines.pt e deeplabv3_roof_pretrain.pt. SEGMENTATION_MODEL_PATH=./models/deeplabv3_roof_multiclass.pt")